In [ ]:
# SLIM WHEEL EXPERIMENT: 9 MB from PyPI instead of 130 MB from the release.
# Viable only if libcifpp accepts a MINIMAL components.cif -- verified locally
# (0.29 MB, 39 components, cpp imports). The open question is whether a full
# fold works, since libcifpp may be consulted when WRITING the output mmCIF.
import os, time, json, subprocess, sys, glob
T0=time.time()
os.system('pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 tokamax==0.0.11 ml_collections')
os.system('pip install -q --no-deps alphafold3-colabfold==3.1.7')
os.system('pip install -q git+https://github.com/sokrypton/py2Dmol.git')
os.system('wget -q -O ccd_fetch.py https://raw.githubusercontent.com/sokrypton/alphafold3/main/src/alphafold3/constants/ccd_fetch.py')
sys.path.insert(0,'.')
import ccd_fetch
codes = ccd_fetch.codes_for_input(extra=('GOL',))
cifs = ccd_fetch.fetch_cifs(codes)
os.makedirs('/content/libcifpp', exist_ok=True)
open('/content/libcifpp/components.cif','w').write('\n'.join(cifs.values()))
# must be set BEFORE anything imports alphafold3, and is inherited by the
# fold subprocess below
os.environ['LIBCIFPP_DATA_DIR']='/content/libcifpp'
import importlib.metadata as md
root=os.path.dirname(md.distribution('alphafold3-colabfold').locate_file('alphafold3'))
conv=os.path.join(root,'alphafold3','constants','converters')
os.makedirs(conv, exist_ok=True)
ccd_fetch.write_pickles(codes, os.path.join(conv,'ccd.pickle'),
                        os.path.join(conv,'chemical_component_sets.pickle'))
os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
os.system('wget -q -O run_alphafold.py https://raw.githubusercontent.com/sokrypton/alphafold3/v3.1.7/run_alphafold.py')
print('SLIM_SETUP_SECONDS: %.0f' % (time.time()-T0))
seq='ACSEFGHIKLWYMNPQRSTV'
json.dump({'dialect':'alphafold3','version':4,'name':'t','modelSeeds':[1],
  'sequences':[{'protein':{'id':'A','sequence':seq,'unpairedMsa':'>q\n'+seq+'\n',
                           'pairedMsa':'','templates':[]}}]}, open('t.json','w'))
env=dict(os.environ, XLA_FLAGS='--xla_disable_hlo_passes=custom-kernel-fusion-rewriter')
t=time.time()
r=subprocess.run([sys.executable,'run_alphafold.py','--json_path=t.json',
  '--model=openbind0','--output_dir=out','--norun_data_pipeline',
  '--num_diffusion_samples=1','--flash_attention_implementation=xla',
  '--buckets=32'],capture_output=True,text=True,env=env)
print('SLIM_FOLD_SECONDS: %.0f rc=%d' % (time.time()-t, r.returncode))
if r.returncode: print((r.stdout+r.stderr)[-1500:])
c=glob.glob('out/**/*model.cif',recursive=True)
print('cifs:',len(c),'atoms:', sum(1 for l in open(c[0]) if l.startswith('ATOM')) if c else 0)
print('SLIM_TOTAL_SECONDS: %.0f' % (time.time()-T0))
print('SLIM_RESULT:', 'PASS' if (r.returncode==0 and c) else 'FAIL')
